In [0]:
SELECT
 COUNT(*)
FROM (SELECT 
    transaction_date,
    product_name,
    destination_city,
    COUNT(*) AS rows_count
FROM supply_chain.silver.silver_supply_chain
GROUP BY transaction_date,
         product_name,
         destination_city
) AS aggregated_data


SELECT * FROM supply_chain.silver.silver_supply_chain;

CREATE TABLE supply_chain.gold.supply_chain_aggregated AS
SELECT 
    transaction_date,
    product_name,
    destination_city,
    ROUND(avg_supplier_reliability_score, 2) AS avg_supplier_reliability_score,
    total_ordered_quantity,
    total_available_inventory,
    ROUND(total_unit_price_usd, 2) AS total_unit_price_usd,
    ROUND(total_product_cost_usd, 2) AS total_product_cost_usd,
    ROUND(total_transportation_cost_usd, 2) AS total_transportation_cost_usd,
    ROUND(total_cost_usd, 2) AS total_cost_usd,
    total_expected_lead_time_days,
    total_actual_lead_time_days,
    delay_days,
    is_delayed,
    is_stockout,
    CURRENT_TIMESTAMP() AS gold_ingestion_timestamp
FROM (
    SELECT 
        transaction_date,
        product_name,
        destination_city,
        product_category,
        AVG(supplier_reliability_score) AS avg_supplier_reliability_score,
        SUM(ordered_quantity) AS total_ordered_quantity,
        SUM(demand_quantity) AS total_demand_quantity,
        SUM(available_inventory) AS total_available_inventory,
        SUM(unit_price_usd) AS total_unit_price_usd,
        SUM(product_cost_usd) AS total_product_cost_usd,
        SUM(transportation_cost_usd) AS total_transportation_cost_usd,
        SUM(total_cost_usd) AS total_cost_usd,
        SUM(expected_lead_time_days) AS total_expected_lead_time_days,
        SUM(actual_lead_time_days) AS total_actual_lead_time_days,
        total_expected_lead_time_days - total_actual_lead_time_days AS delay_days,
        CASE
            WHEN (total_expected_lead_time_days - total_actual_lead_time_days)  < 0 THEN 0
            ELSE 1
        END AS is_delayed,
        CASE
            WHEN total_ordered_quantity = total_available_inventory THEN 0
            ELSE 1
        END AS is_stockout
    FROM supply_chain.silver.silver_supply_chain
    GROUP BY transaction_date,
            product_name,
            destination_city,
            product_category
) AS aggregated_data;


SELECT * FROM supply_chain.gold.supply_chain_aggregated;
DROP TABLE supply_chain.gold.aggregated_supply_chain;
SELECT COUNT(*)
FROM supply_chain.gold.supply_chain_aggregated;


SELECT
    COUNT(*)
FROM (SELECT 
        transaction_date,
        product_name,
        destination_city,
        product_category,
        AVG(supplier_reliability_score) AS avg_supplier_reliability_score,
        SUM(ordered_quantity) AS total_ordered_quantity,
        SUM(demand_quantity) AS total_demand_quantity,
        SUM(available_inventory) AS total_available_inventory,
        SUM(unit_price_usd) AS total_unit_price_usd,
        SUM(product_cost_usd) AS total_product_cost_usd,
        SUM(transportation_cost_usd) AS total_transportation_cost_usd,
        SUM(total_cost_usd) AS total_cost_usd,
        SUM(expected_lead_time_days) AS total_expected_lead_time_days,
        SUM(actual_lead_time_days) AS total_actual_lead_time_days,
        total_expected_lead_time_days - total_actual_lead_time_days AS delay_days,
        CASE
            WHEN (total_expected_lead_time_days - total_actual_lead_time_days)  < 0 THEN 0
            ELSE 1
        END AS is_delayed,
        CASE
            WHEN total_ordered_quantity = total_available_inventory THEN 0
            ELSE 1
        END AS is_stockout
    FROM supply_chain.silver.silver_supply_chain
    GROUP BY transaction_date,
            product_name,
            destination_city,
            product_category
)

SELECT
    COUNT(*)
FROM (SELECT
    COUNT(*)
FROM supply_chain.silver.silver_supply_chain
    GROUP BY product_name,
            destination_city,
            product_category
)

CREATE OR REPLACE supply_chain.gold.feature_supply_chain AS
SELECT 
    *,
    ROUND(LAG(total_demand_quantity) OVER (PARTITION BY product_name, destination_city ORDER BY transaction_date), 2) AS previouse_date_total_demand,
    ROUND(SUM(total_demand_quantity) OVER(PARTITION BY product_name, destination_city ORDER BY transaction_date), 2) AS cummulative_sum
FROM supply_chain.gold.aggregated_supply_chain;

















